In [1]:
!apt-get update -y
!apt-get install -y curl
!pip install pandas scikit-learn regex

# --- Install Ollama (optional; may not work on Colab GPU runtimes) ---
!curl -fsSL https://ollama.com/install.sh | sh || echo "⚠️ Ollama install may not work in Colab (requires local environment)"
!pip install tldextract

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:8 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,525 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [2]:
def get_topic_specific_metrics(topic):
    """
    Return evaluation metrics based on the exact topic category you provided.
    """
    topic = topic.lower().strip()
    topic_rules = {
        "arts": """
            - Verify artist/institution names against known databases
            - Check exhibition dates and locations for accuracy
            - Flag subjective superlatives (e.g., "greatest", "unprecedented")
            - Validate cultural event details
            - Detect fabricated art market prices
        """,

        "crime": """
            - Cross-check location, date, and victim details
            - Verify police/court official statements
            - Flag unattributed crime statistics
            - Detect sensationalist crime language (e.g., "horrific", "bloodbath")
            - Check for named suspects/victims without official confirmation
            - Validate legal terminology accuracy
        """,

        "disaster and accident": """
            - Verify casualty numbers with official sources
            - Check geographic location accuracy
            - Flag fear-mongering language (e.g., "catastrophic", "apocalyptic")
            - Validate emergency response details
            - Cross-reference timeline with known events
            - Detect exaggerated damage estimates
        """,

        "economy": """
            - Verify numerical data (GDP, inflation, exchange rates)
            - Validate company/institution names
            - Check economic terminology correctness
            - Flag unattributed market predictions
            - Detect misleading percentage comparisons
            - Verify government policy references
            - Check for unrealistic growth/loss claims
        """,

        "education": """
            - Verify educational institution existence
            - Validate exam results and statistics
            - Check ministry/government policy references
            - Flag fabricated reform announcements
            - Detect misleading international ranking claims
            - Verify scholarship/funding program details
        """,

        "environment": """
            - Validate scientific terminology and data
            - Cross-check climate statistics with research
            - Flag alarmist language without evidence
            - Verify environmental organization citations
            - Detect pseudoscience or conspiracy theories
            - Check consistency with scientific consensus
        """,

        "health": """
            - Verify medical terminology accuracy
            - Flag miracle cure or panic-inducing claims
            - Check for WHO/health ministry citations
            - Detect pseudoscience or alternative medicine fraud
            - Validate drug/treatment names
            - Cross-reference medical statistics
            - Flag unverified health advice
        """,

        "human interest": """
            - Assess emotional manipulation level
            - Check narrative consistency and plausibility
            - Verify named individuals (if public figures)
            - Flag overly dramatic storytelling
            - Detect fabricated heartwarming/tragic stories
            - Check for privacy violations
        """,

        "labour": """
            - Verify union/organization names
            - Check employment statistics accuracy
            - Validate labour law references
            - Flag exaggerated strike/protest numbers
            - Cross-reference wage/unemployment data
            - Detect biased pro-employer or pro-worker language
        """,

        "lifestyle and leisure": """
            - Identify clickbait headlines
            - Distinguish opinion from factual reporting
            - Verify brand/celebrity names
            - Check for undisclosed advertising
            - Flag sensationalized lifestyle trends
            - Detect fabricated social media trends
        """,

        "politics": """
            - Verify politician names, titles, and parties
            - Check for propaganda indicators
            - Detect political bias or loaded language
            - Validate government policy references
            - Cross-check political event details
            - Flag unattributed quotes from officials
            - Detect conspiracy theories
            - Verify electoral/polling data
        """,

        "religion and belief": """
            - Validate religious institution names
            - Check for extremist language indicators
            - Verify religious scholar attributions
            - Detect fabricated fatwas or religious rulings
            - Flag sectarian bias or hate speech
            - Validate religious event dates and locations
        """,

        "science and technology": """
            - Verify scientific/technical terminology
            - Check for peer-reviewed research citations
            - Flag claims contradicting established science
            - Detect exaggerated tech breakthrough claims
            - Validate researcher/institution names
            - Cross-check with scientific databases
            - Flag pseudoscience indicators
        """,

        "society": """
            - Verify social statistics and surveys
            - Check named organizations/institutions
            - Flag stereotyping or generalization
            - Detect fabricated social trends
            - Validate demographic data
            - Cross-check cultural event details
        """,

        "sport": """
            - Verify team names, players, and officials
            - Check match results and scores
            - Validate sporting event dates/locations
            - Cross-check tournament participants with official lists
            - Flag transfer rumors without sources
            - Detect exaggerated performance claims
            - If non-eligible teams appear in tournament groups (e.g., non-African teams in AFCON), classify as Fake immediately
            - Reject any group lists that contradict the official tournament structure
        """,

        "war": """
            - Verify military terminology and locations
            - Check casualty figures against official sources
            - Detect propaganda language
            - Flag unverified combat reports
            - Validate military official statements
            - Cross-check with international news sources
            - Detect emotional manipulation tactics
        """,

        "weather": """
            - Verify meteorological terminology
            - Check temperature/precipitation accuracy
            - Flag catastrophic language without data
            - Validate weather service citations
            - Cross-reference predictions with official forecasts
            - Detect seasonal impossibilities
        """,

        "unknown": """
            - Apply general credibility assessment
            - Check linguistic consistency
            - Verify named entities when possible
            - Flag sensationalist language
            - Assess logical coherence
            - Check for source attribution
        """
    }

    return topic_rules.get(topic, """
        - Apply general linguistic, factual, and sentiment-based metrics
        - Flag missing source attribution
        - Check for logical inconsistencies
        - Detect sensationalist language patterns
    """)
TRUSTED_DOMAINS = {
    "youm7.com",
    "elwatannews.com",
    "almasryalyoum.com",
    "elbalad.news",
    "masrawy.com",
    "vetogate.com",
    "dostor.org",
    "egypttoday.com",
    "shorouknews.com",
    "cairo24.com",
    "akhbarelyom.com",
    "ahram.org.eg",
    "english.ahram.org.eg",
    "dailynewsegypt.com",
    "egyptianstreets.com",
    "madamasr.com",
    "aldostor.com",
    "mobtada.com",
    "albawabhnews.com",
    "almalnews.com",
    "alborsanews.com",
    "amwalalghad.com",
    "alwafd.news",
    "gomhuriaonline.com",
    "rosaelyoussef.com",
    "see.news",
    "egyptindependent.com",
    "egyptian-gazette.com",
    "egyptbusiness.com",
    "enterprise.press",
    "egyptoil-gas.com",
    "cairoscene.com",
    "thestartupscene.me",
    "cairo360.com",
    "watani.net",
    "egynews.net",
    "nileinternational.net",
    "filgoal.com",
    "yallakora.com",
    "kingfut.com",
    "misrday.com",
    "elconsolto.com",
    "reuters.com",
    "apnews.com",
    "afp.com",
    "bbc.com",
    "aljazeera.com",
    "aljazeera.net",
    "arabnews.com",
    "thenationalnews.com",
    "aawsat.com",
    "alarabiya.net",
    "skynewsarabia.com",
    "dw.com",
    "france24.com",
    "rfi.fr",
    "cnn.com",
    "nytimes.com",
    "washingtonpost.com",
    "ft.com",
    "bloomberg.com",
    "wsj.com",
    "economist.com",
    "middleeasteye.net",
    "allafrica.com",
    "theafricareport.com",
    "zawya.com",
    "oxfordbusinessgroup.com",
    "africa-confidential.com",
    "africanews.com",
    "elmogaz.com",
    "innfrad.com",
    "soutalomma.com",
    "elmostaqbal.com",
    "akhbarak.net",
    "masress.com",
    "elbashayer.com",
    "nogoumfm.net",
    "elgornal.net",
    "elaosboa.com",
    "elzmannews.com",
    "masralarabia.com",
    "rassd.com",
    "baladnaelyoum.com",
    "elmwatin.com",
    "alnaharegypt.com",
    "altyaargate.com",
    "parlmany.com",
    "sada-elarab.com",
    "albawaba.com",
    "middleeastmonitor.com",
    "egyptdailynews.com",
    "copts-united.com",
    "arabic.cnn.com",
    "independentarabia.com",
    "asharq.com",
    "bloombergasharq.com",
    "mubasher.info",
    "egx.com.eg",
    "cbe.org.eg",
    "capmas.gov.eg",
    "sis.gov.eg",
    "mfa.gov.eg",
    "mohp.gov.eg"
}


In [3]:
import requests
import re
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, unquote
MAX_QUERY_CHARS = 300  # safe for search engines

def safe_search_query(text):
    return text[:MAX_QUERY_CHARS]

def extract_real_url(ddg_url):
    parsed = urlparse(ddg_url)
    qs = parse_qs(parsed.query)
    if "uddg" in qs:
        return unquote(qs["uddg"][0])
    return ddg_url


def scrape_text(url):
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "en-US,en;q=0.9"
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.RequestException:
        return ""

    soup = BeautifulSoup(response.text, "lxml")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines)


def clean_text(text):
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.replace("|", "")  # Remove all | characters
    return text.strip()
MAX_CHARS = 4000

def safe_text(text):
    return text[:MAX_CHARS]


# =========================
# MAIN FUNCTION YOU NEED
# =========================

def search_and_extract_facts(search_query, max_facts=5):

    # ---- DuckDuckGo Search ----
    url = "https://duckduckgo.com/html"
    params = {"q": safe_search_query(search_query)}
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, params=params, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    results = soup.select(".result__body")

    search_results = []

    for r in results:
        title_tag = r.select_one(".result__title a")
        if not title_tag:
            continue

        real_link = extract_real_url(title_tag["href"])
        domain = urlparse(real_link).netloc

        trusted = 1 if any(td in domain for td in TRUSTED_DOMAINS) else 0

        search_results.append({
            "url": real_link,
            "trusted": trusted
        })

    # ---- Trusted Stats ----
    trusted_count = sum(r["trusted"] for r in search_results)
    trusted_exists = trusted_count > 0

    # ---- Extract Facts ----
    facts_list = []
    for r in search_results[:max_facts]:
        content = scrape_text(r["url"])
        if content:
            facts_list.append(safe_text(clean_text(content)))
    facts = "|".join(facts_list)
    return facts, trusted_count, trusted_exists



In [4]:

import ast

def check_count(lst):
    return 10 if lst.count(1) >= 3 else 0
def parse_llm_list(text: str):
    """
    Extracts a Python list like [1, 0, 1, 1, 0] from LLM output safely.
    """
    match = re.search(r"\[\s*[01](?:\s*,\s*[01]){4}\s*\]", text)
    if not match:
        return None

    try:
        return ast.literal_eval(match.group())
    except Exception:
        return None
import re

def clean_topic(raw_response):
    VALID_TOPICS = [
        "Arts", "Crime", "Disaster and Accident", "Economy", "Education",
        "Environment", "Health", "Human Interest", "Labour",
        "Lifestyle and Leisure", "Politics", "Religion and Belief",
        "Science and Technology", "Society", "Sport", "War",
        "Weather", "Unknown"
    ]

    # Convert to string and lowercase for matching
    text = str(raw_response).lower()

    # Check each valid topic (case-insensitive)
    for topic in VALID_TOPICS:
        # Create pattern that matches the topic word
        pattern = r'\b' + re.escape(topic.lower()) + r'\b'
        if re.search(pattern, text):
            return topic

    # If no match found, return "Unknown"
    return "Unknown"


In [5]:
import re
import json
import time
import subprocess
import tldextract

In [6]:
MODEL_NAME = "gemma3:4b"  # Ensure you pull this model first
#MODEL_NAME="llama2:7b"
# Weighting Logic (Calculated in Python, not LLM)
METRIC_WEIGHTS = {
     "source_reputation": 0.15, # Python-checked domain trust
        "evidence_quality": 0.15,
    "fact_check": 0.45, # Does the logic hold up?
    "topic_consistency": 0.05,  # Does it match the topic context?
    "cross_reference": 0.10 ,
    "writing_style": 0.02 , # Is it professional/sensational?
  "topic_rules": 0.08
}

def extract_json(text):
    """Robustly extracts JSON object from LLM chatter."""
    # Find the first { and the last }
    match = re.search(r"(\{.*\})", text, re.DOTALL)
    if match:
        json_str = match.group(1)
        # Cleanup common LLM JSON errors
        json_str = re.sub(r",\s*}", "}", json_str) # Trailing commas
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            return None
    return None

def run_ollama(prompt):
    """Sends prompt to Ollama and returns raw string response."""
    cmd = [
        "curl", "-s", "-X", "POST", "http://localhost:11434/api/generate",
        "-H", "Content-Type: application/json",
        "-d", json.dumps({"model": MODEL_NAME, "prompt": prompt, "stream": False})
    ]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        response_json = json.loads(result.stdout)
        return response_json.get("response", "")
    except Exception as e:
        print(f"Error calling Ollama: {e}")
        return ""
def classify_news(article_text: str, trusted_count, trusted_exists, facts):
    try:
        subprocess.Popen(["ollama", "serve"])
        time.sleep(5)
        subprocess.run(["ollama", "pull", MODEL_NAME], check=True)
    except Exception as e:
        return {
             "topic": f"Ollama unavailable: {e}",
        "fact_check_score": None,
        "classification":None,
        "confidence_score": None
        }

    source_score = 10 if trusted_exists else 5
    cross_reference = 10 if trusted_count >= 2 else 5

    # STEP 1 – Detect Topic

    topic_prompt = f"""
    Determine the main topic of the following Arabic article.
    Choose strictly from:
    ["Arts","Crime","Disaster and Accident","Economy","Education","Environment",
    "Health","Human Interest","Labour","Lifestyle and Leisure","Politics",
    "Religion and Belief","Science and Technology","Society","Sport","War","Weather","Unknown"]
    Article:
    {article_text}
    """
    raw_topic = run_ollama(topic_prompt)
    topic = raw_topic.strip() if raw_topic else "Unknown"

    topic_rules = get_topic_specific_metrics(topic)

    # STEP 2 - Fact Check
    facts_list = facts.split("|")
    # Safe access to facts list indices
    f1 = facts_list[0] if len(facts_list) > 0 else "N/A"
    f2 = facts_list[1] if len(facts_list) > 1 else "N/A"
    f3 = facts_list[2] if len(facts_list) > 2 else "N/A"
    f4 = facts_list[3] if len(facts_list) > 3 else "N/A"
    f5 = facts_list[4] if len(facts_list) > 4 else "N/A"

    prompt = f"""
    SYSTEM: You are a JSON-only fact verification API. You must return ONLY valid JSON. No explanations allowed.

    TASK: Compare facts against article text and return verification scores.

    ARTICLE TEXT:
    {article_text}

    FACTS TO VERIFY:
    1. {f1}
    2. {f2}
    3. {f3}
    4. {f4}
    5. {f5}

    RULES:
    - 1 = fact is directly stated in article with matching numbers/values
    - 0 = fact contradicts article or is not mentioned

    REQUIRED OUTPUT FORMAT (nothing else):
    {{"verification_results": [0, 0, 0, 0, 0]}}
    """

    fact_check_response = run_ollama(prompt)
    print("🔍 Fact Check Response:", fact_check_response)

    verification_list = [0, 0, 0, 0, 0]
    fact_check_score = 0

    if not fact_check_response:
        print("❌ Failed to extract verification JSON")
    else:
        try:
            parsed = extract_json(fact_check_response) # Used robust extractor here too
            if parsed and "verification_results" in parsed:
                verification_list = parsed["verification_results"]
                if len(verification_list) != 5:
                    verification_list = (verification_list + [0, 0, 0, 0, 0])[:5]
                fact_check_score = check_count(verification_list)
            else:
                 print("❌ JSON missing 'verification_results' key")
        except Exception:
            print("❌ Invalid JSON returned from Ollama")
    # STEP 3 - Other Metrics
    prompt = f"""
    Analyze this news text and return ONLY valid JSON with these 4 scores (0-10):
    1. writing_style: Score 10 for neutral, professional journalism. Score 0 for excessive emotional manipulation, multiple exclamation marks!!!, or hate speech.
    2. evidence_quality: Score 10 if it cites specific names, dates, and official sources. Score 0 for vague attributions ("sources said").
    3. topic_consistency: Does the text stay on topic?
    4. topic_rules :{topic_rules}
    TEXT:
    {article_text}

    Output EXACTLY this format:
    {{
      "writing_style": 7,
      "evidence_quality": 8,
      "topic_consistency": 6,
      "topic_rules": 5,
      "flag_reason": "Brief explanation"
    }}

    Output ONLY the JSON:
    """
    raw_response = run_ollama(prompt)

    metrics = extract_json(raw_response)

    # FIX: Ensure all metrics are integers, not dicts
    if not metrics:
        metrics = {
            "writing_style": 5,
            "evidence_quality": 5,
            "topic_consistency": 5,
            "topic_rules": 5,
            "flag_reason": "Model failed to generate valid JSON"
        }
    else:
        # Ensure each metric is an integer
        for key in ["writing_style", "evidence_quality", "topic_consistency", "topic_rules"]:
            value = metrics.get(key, 5)
            # If it's a dict or any non-numeric type, default to 5
            if not isinstance(value, (int, float)):
                print(f"⚠️ Warning: {key} is not a number: {value}")
                metrics[key] = 5
            else:
                metrics[key] = int(value)  # Ensure it's an integer

    # Calculate Final Weighted Score
    weighted_score = (
        (source_score * METRIC_WEIGHTS["source_reputation"]) +
        (fact_check_score * METRIC_WEIGHTS["fact_check"]) +
        (metrics["writing_style"] * METRIC_WEIGHTS["writing_style"]) +
        (metrics["evidence_quality"] * METRIC_WEIGHTS["evidence_quality"]) +
        (metrics["topic_consistency"] * METRIC_WEIGHTS["topic_consistency"]) +
        (metrics["topic_rules"] * METRIC_WEIGHTS["topic_rules"]) +
        (cross_reference * METRIC_WEIGHTS["cross_reference"])
    )

    final_percentage = round(weighted_score * 10, 2)
    classification = "REAL" if final_percentage >= 70 else "FAKE" # 65

    return {
        "topic": topic,
        "fact_check_score": fact_check_score,
        "classification": classification,
        "confidence_score": f"{final_percentage}%"
    }

In [7]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
from pandas.core.tools.datetimes import to_datetime
import os
import pandas as pd
def process_file(input_path, output_path):
    df = pd.read_csv(input_path).head(10)

    # Initialize new columns
    df["topic"] = ""
    df["classification"] = ""
    df["confidence_score"] = ""
    df["fact_check_score"] = ""

    print(f"🚀 Classifying {input_path} ...")

    for idx, text in df["Article_Text"].items():
        if text == "Not found":
           text = df.at[idx, "Title"]
           print(text)
        facts, trusted_count, trusted_exists = search_and_extract_facts(
          safe_search_query(text), max_facts=5
        )

        r = classify_news(text, trusted_count, trusted_exists, facts)

        # Optional: print response for debugging

        print(json.dumps({
            "topic": r.get("topic", ""),
            "classification": r.get("classification", ""),
            "fact_check_score": r.get("fact_check_score", ""),
            "confidence_score": r.get("confidence_score", "")  # Changed from final_percentage
        }, indent=2, ensure_ascii=False))

        # Safely extract values from LLM response
        df.at[idx, "topic"] = clean_topic(r.get("topic", ""))
        df.at[idx, "classification"] = r.get("classification", "")
        df.at[idx, "confidence_score"] = r.get("confidence_score", "")  # Fixed
        df.at[idx, "fact_check_score"] = r.get("fact_check_score", "")
    # Save final dataframe
    df.to_csv(output_path, index=False)
    print(f"✅ Saved results to {output_path}")

files = [
    ("/content/drive/MyDrive/news_data/elshrouk.csv", "/content/drive/MyDrive/classified_data/elshrouk_reality.csv"),
    ("/content/drive/MyDrive/news_data/elmasrielyoum.csv", "/content/drive/MyDrive/classified_data/elmasrielyoum_reality.csv"),
    ("/content/drive/MyDrive/news_data/elwatan.csv", "/content/drive/MyDrive/classified_data/elwatan_reality.csv"),
    ("/content/drive/MyDrive/news_data/masrawy.csv", "/content/drive/MyDrive/classified_data/masrawy_reality.csv"),
    ("/content/drive/MyDrive/news_data/youm7.csv", "/content/drive/MyDrive/classified_data/youm7_reality.csv"),
    ("/content/drive/MyDrive/news_data/elbashayer.csv", "/content/drive/MyDrive/classified_data/elbashayer_reality.csv")
]

for inp, outp in files:
    if os.path.exists(inp):
        df = process_file(inp, outp)
    else:
        print(f"⚠️ Skipping missing file: {inp}")


🚀 Classifying /content/drive/MyDrive/news_data/elshrouk.csv ...
🔍 Fact Check Response: ```json
{"verification_results": [1, 0, 0, 0, 0]}
```
{
  "topic": "The main topic of the article is **Disaster and Accident**.\n\nHere's why:\n\nThe article details a series of tragic incidents involving patients dying while waiting for critical care beds in hospitals, highlighting a severe shortage and a system failure that results in preventable deaths. The focus is squarely on the disastrous outcomes of delays and lack of access to necessary medical care.",
  "classification": "FAKE",
  "fact_check_score": 0,
  "confidence_score": "32.9%"
}
🔍 Fact Check Response: ```json
{"verification_results": [1, 1, 1, 1, 1]}
```
{
  "topic": "Sport",
  "classification": "REAL",
  "fact_check_score": 10,
  "confidence_score": "74.6%"
}
🔍 Fact Check Response: ```json
{"verification_results": [1, 1, 1, 1, 1]}
```
{
  "topic": "Economy",
  "classification": "REAL",
  "fact_check_score": 10,
  "confidence_score": 